# Gravitino: one metadata layer, Spark and Trino interchangeably

This notebook shows Gravitino acting as a single metadata layer across two engines and two storage backends. We write data into a Hive table and an Iceberg table with **Spark**, query them with **Trino**, and finish with a federated Trino query that unions Hive and Iceberg data through Gravitino, no per-engine catalog wiring required.

## Write a Hive table with Spark

Start a Spark session configured with the Gravitino Spark connector. The connector points Spark at Gravitino (`spark.sql.gravitino.uri`) so Spark sees Gravitino's catalogs directly.

In [ ]:
import pyspark
import os
from pyspark.sql import SparkSession

spark_home = os.getenv('SPARK_HOME')
gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME'] = "anonymous"

spark = SparkSession.builder \
    .appName("PySpark SQL Example") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/packages/iceberg-spark-runtime-3.4_2.12-1.10.0.jar,/tmp/gravitino/packages/{gravitino_connector_jar}") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.gravitino.enableIcebergSupport", "true") \
    .config("spark.sql.catalog.catalog_rest", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.catalog_rest.type", "rest") \
    .config("spark.sql.catalog.catalog_rest.uri", "http://gravitino:9001/iceberg/") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .enableHiveSupport() \
    .getOrCreate()

Select the Hive catalog and list its databases. `catalog_hive` is one of the catalogs the playground registers in Gravitino.

In [ ]:
spark.sql("use catalog_hive")
spark.sql("show databases").show()

Create a `product` database and a partitioned `employees` table, then describe it to confirm the schema and partitioning.

In [ ]:
spark.sql("DROP TABLE IF EXISTS catalog_hive.product.employees")
spark.sql("DROP DATABASE IF EXISTS catalog_hive.product CASCADE")

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS product;")
spark.sql("USE product;")
spark.sql("CREATE TABLE IF NOT EXISTS employees (id INT, name STRING, age INT) PARTITIONED BY (department STRING) STORED AS PARQUET;")
spark.sql("DESC TABLE EXTENDED employees;").show()

Insert rows into two partitions (`Engineering` and `Marketing`), then read the table back.

In [ ]:
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Engineering') VALUES (1, 'John Doe', 30), (2, 'Jane Smith', 28);")
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Marketing') VALUES (3, 'Mike Brown', 32);")
spark.sql("SELECT * from employees").show()

## Query the Hive table with Trino

The same table written by Spark is now queryable through Trino, because both engines share Gravitino's metadata. Install the Trino client and pandas (used to render results as a table), then connect.

In [ ]:
%pip install -q pandas
%pip install -q trino==0.335.0

In [ ]:
from trino.dbapi import connect
import pandas as pd

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="catalog_hive",
    schema="product",
)
trino_client = conn.cursor()

Query the Engineering employees. We render the result as a pandas DataFrame so it reads as a proper table rather than a raw list of tuples.

In [ ]:
rows = trino_client.execute(
    "SELECT * FROM catalog_hive.product.employees WHERE department = 'Engineering'").fetchall()
pd.DataFrame(rows, columns=['id', 'name', 'age', 'department'])

## Write an Iceberg table via the Iceberg REST service

Now switch to `catalog_rest`, an Iceberg catalog served by Gravitino's Iceberg REST endpoint. We create a `sales.customers` table and insert rows, all through Spark.

In [ ]:
spark.sql("use catalog_rest;")
spark.sql("create database if not exists sales;")
spark.sql("use sales;")
spark.sql("create table if not exists customers (customer_id int, customer_name string, customer_email string);")

In [ ]:
spark.sql("insert into customers (customer_id, customer_name, customer_email) values (11,'Rory Brown','rory@123.com');")
spark.sql("insert into customers (customer_id, customer_name, customer_email) values (12,'Jerry Washington','jerry@dt.com');")
spark.sql("select * from customers").show()

## Federated query across Hive and Iceberg with Trino

The payoff: a single Trino query that unions data from the Hive catalog and the Iceberg catalog. Because Gravitino federates both under one metadata layer, Trino can join across them with no extra configuration. We render the combined result as a DataFrame.

In [ ]:
rows = trino_client.execute(
    "SELECT * FROM catalog_hive.sales.customers "
    "UNION SELECT * FROM catalog_iceberg.sales.customers").fetchall()
pd.DataFrame(rows, columns=['customer_id', 'customer_name', 'customer_email'])